# 02c – Unpivot dữ liệu lâm sàng (tùy chọn)

Mục đích: chuyển file điều trị lâm sàng do giảng viên cung cấp (oxy, kháng sinh tại nhập viện / 24 giờ / 48 giờ, số giờ thở máy) từ dạng rộng sang dạng dài (mục 3.4). Notebook này trước đây tên `01b_clinical_unpivot_optional`; đã đổi thành `02c` vì `01b` thuộc về EDA Synthea.

**Trạng thái: chưa có file thật.** Notebook được viết theo mô tả và kiểm thử trên dữ liệu giả dựng tại chỗ (ô 2b), không phải trên dữ liệu của giảng viên. Khi có file thật:
1. Đặt file vào `data/raw/`, sửa `CLINICAL_FILE`, `HEADER_ROWS`.
2. Chạy lại, đọc phần báo cáo (cột không nhận diện được, tên thuốc và tên oxy chưa khớp, hàng mâu thuẫn) rồi sửa các từ điển ở ô cấu hình.
3. Hỏi giảng viên ô trống nghĩa là gì (chưa ghi nhận hay không dùng thuốc) rồi chỉnh `NO_DRUG_TOKENS`, `MISSING_TOKENS`.

Đầu ra (`data/anonymized/`): `clinical_long.csv` (mỗi bệnh nhân đúng 3 dòng), `clinical_antibiotic_bridge.csv` (mỗi thuốc một dòng), `clinical_drug_names.csv`, `clinical_ventilation.csv`, `clinical_conflicts.csv`.

In [ ]:
import sys, re, difflib
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                            if (p / 'src' / 'anon_common.py').exists()) / 'src'))
import numpy as np
import pandas as pd
from anon_common import RAW, ANON, get_salt, pseudonymize

SALT = get_salt()

# ---- Cấu hình: sửa theo file thật ----
CLINICAL_FILE = RAW / 'clinical_treatment.xlsx'     # tên file giả định
HEADER_ROWS = [0, 1]                                # số dòng tiêu đề (2 = tiêu đề hai tầng)

PATIENT_COL_RE = r'patient|mã\s*b|ma\s*bn|^id\b|\bid\b'
VENT_COL_RE = r'vent|thở máy|tho may'
ABX_FLAG_RE = r'sử dụng kháng sinh|su dung khang sinh|antibiotic use|abx use'   # cột Có/Không riêng (nếu có)
TIMEPOINTS = {'admission': r'admi|nhập viện|nhap vien|\b0\s*h', '24h': r'24', '48h': r'48'}
NO_DRUG_TOKENS = {'none', 'no', 'nil', 'không', 'khong', 'no antibiotic', 'không dùng'}
YES_TOKENS = {'có', 'co', 'yes', 'y', '1', 'true'}
MISSING_TOKENS = {'', 'nan', 'na', 'n/a', 'null', 'unknown', '?', 'chưa rõ', 'chua ro'}

CANONICAL_DRUGS = ['Ertapenem', 'Meropenem', 'Imipenem', 'Linezolid', 'Metronidazole', 'Vancomycin',
                   'Ceftriaxone', 'Cefepime', 'Piperacillin-Tazobactam', 'Azithromycin', 'Levofloxacin',
                   'Ciprofloxacin', 'Colistin', 'Amoxicillin-Clavulanate', 'Doxycycline']
DRUG_ALIASES = {'etarpenem': 'Ertapenem', 'linezolide': 'Linezolid', 'metronidazol': 'Metronidazole',
                'piperacillin/tazobactam': 'Piperacillin-Tazobactam', 'pip-tazo': 'Piperacillin-Tazobactam',
                'augmentin': 'Amoxicillin-Clavulanate'}
# Bí danh chỉ là SUY ĐOÁN lỗi chính tả, chưa được giảng viên xác nhận: ghi nhãn riêng để người xem duyệt.
UNCONFIRMED_ALIASES = {'etarpenem'}
FUZZY_CUTOFF = 0.85
# Tên oxy tiếng Việt và viết tắt: bản khởi đầu, cần bổ sung khi có file thật
OXYGEN_ALIASES = {
    'ra': 'Room air', 'room air': 'Room air', 'khí phòng': 'Room air', 'khi phong': 'Room air',
    'nasal cannula': 'Nasal cannula', 'nc': 'Nasal cannula', 'canuyn mũi': 'Nasal cannula',
    'mask': 'Face mask', 'face mask': 'Face mask', 'mặt nạ': 'Face mask',
    'hfnc': 'High-flow nasal cannula', 'niv': 'Non-invasive ventilation', 'bipap': 'Non-invasive ventilation',
    'intubated': 'Invasive ventilation', 'mechanical ventilation': 'Invasive ventilation',
    'thở máy': 'Invasive ventilation', 'tho may': 'Invasive ventilation',
    'thở oxy': 'Oxygen (unspecified)', 'tho oxy': 'Oxygen (unspecified)',
}

## 1. Hàm xử lý

In [ ]:
def flatten_columns(df):
    """Làm phẳng tiêu đề nhiều tầng: điền tiếp tầng trên cho ô 'Unnamed', nối các tầng bằng ' | ', bỏ tầng rỗng."""
    if not isinstance(df.columns, pd.MultiIndex):
        df.columns = [str(c).strip() for c in df.columns]
        return df
    levels = pd.DataFrame(df.columns.tolist())
    levels = levels.apply(lambda col: col.map(lambda x: np.nan if str(x).startswith('Unnamed') else str(x).strip()))
    levels[0] = levels[0].ffill()                      # tầng trên bị gộp ô → điền tiếp
    df.columns = [' | '.join(dict.fromkeys(str(x) for x in row if pd.notna(x) and str(x).strip())) for row in levels.values]
    return df

def classify_value(raw):
    """Phân biệt: 'missing' (không ghi nhận), 'none' (ghi rõ không dùng), 'given' (có dùng)."""
    if pd.isna(raw): return 'missing'
    t = str(raw).strip().lower()
    if t in MISSING_TOKENS: return 'missing'
    if t in NO_DRUG_TOKENS: return 'none'
    return 'given'

def normalize_drug(name):
    """Trả về (tên chuẩn, cách khớp). Không đoán bừa: dưới ngưỡng thì giữ nguyên và đánh dấu 'unmatched'."""
    key = re.sub(r'\s+', ' ', name.strip().lower())
    if key in DRUG_ALIASES:
        return DRUG_ALIASES[key], ('alias_unconfirmed' if key in UNCONFIRMED_ALIASES else 'alias')
    for c in CANONICAL_DRUGS:
        if key == c.lower():
            return c, 'exact'
    hit = difflib.get_close_matches(key, [c.lower() for c in CANONICAL_DRUGS], n=1, cutoff=FUZZY_CUTOFF)
    if hit:
        return next(c for c in CANONICAL_DRUGS if c.lower() == hit[0]), 'fuzzy'
    return name.strip().title(), 'unmatched'

def normalize_support(name):
    key = re.sub(r'\s+', ' ', name.strip().lower())
    if key in OXYGEN_ALIASES:
        return OXYGEN_ALIASES[key], 'alias'
    return name.strip(), 'unmatched'

def split_plus(raw):
    """Tách theo dấu '+': 'Linezolide + Metronidazol' thành hai thuốc; 'HFNC+ NIV' thành hai hình thức."""
    return [p.strip() for p in str(raw).split('+') if p.strip()]

MATCH_RANK = ['unmatched', 'alias_unconfirmed', 'fuzzy', 'alias', 'exact', 'ok']     # từ kém tin cậy nhất

def worst_match(methods):
    return min(methods, key=MATCH_RANK.index) if methods else None

def timepoint_of(col):
    for tp, rx in TIMEPOINTS.items():
        if re.search(rx, col.lower()):
            return tp
    return None

def find_columns(df, topic_rx, exclude=()):
    """Các cột thuộc một chủ đề (oxy / kháng sinh) kèm mốc thời gian của chúng."""
    found, unknown = {}, []
    for c in df.columns:
        if c in exclude or not re.search(topic_rx, c.lower()):
            continue
        tp = timepoint_of(c)
        (found.__setitem__(tp, c) if tp else unknown.append(c))
    return found, unknown

## 2. Quy trình chính

Đầu ra theo yêu cầu nghiệm thu của đề:
- `clinical_long.csv`: **mỗi bệnh nhân đúng 3 dòng** (admission, 24h, 48h), mỗi dòng có hình thức oxy và phác đồ kháng sinh đã chuẩn hóa.
- `clinical_antibiotic_bridge.csv`: **bảng cầu, mỗi thuốc một dòng** (bệnh nhân, mốc, thuốc).
- `clinical_drug_names.csv`: danh sách tên thuốc đã chuẩn hóa, kèm cách khớp và các cách viết gốc.

In [ ]:
TP_ORDER = ['admission', '24h', '48h']

def process(df):
    df = flatten_columns(df.copy())
    pid_col = next(c for c in df.columns if re.search(PATIENT_COL_RE, c.lower()))
    flag_col = next((c for c in df.columns if re.search(ABX_FLAG_RE, c.lower())), None)
    oxy_cols, oxy_unknown = find_columns(df, r'oxy|oxi|thở oxy')
    abx_cols, abx_unknown = find_columns(df, r'antibio|kháng sinh|khang sinh|abx', exclude=(flag_col,))
    vent_col = next((c for c in df.columns if re.search(VENT_COL_RE, c.lower())), None)
    pid = df[pid_col].astype(str).map(pseudonymize)

    long_rows, bridge_rows = [], []
    for i in range(len(df)):
        for tp in TP_ORDER:                                          # đúng 3 dòng mỗi bệnh nhân
            ov = df[oxy_cols[tp]].iloc[i] if tp in oxy_cols else np.nan
            av = df[abx_cols[tp]].iloc[i] if tp in abx_cols else np.nan
            o_status, a_status = classify_value(ov), classify_value(av)
            o_label, o_match = None, None
            if o_status == 'given':
                parts = [normalize_support(p) for p in split_plus(ov)]
                o_label = ' + '.join(l for l, _ in parts)
                o_match = worst_match(['unmatched' if h == 'unmatched' else 'ok' for _, h in parts])
            a_label, a_match = None, None
            if a_status == 'given':
                drugs = [(raw, *normalize_drug(raw)) for raw in split_plus(av)]
                a_label = ' + '.join(std for _, std, _ in drugs)
                a_match = worst_match([h for _, _, h in drugs])
                for raw, std, how in drugs:                          # bảng cầu: mỗi thuốc một dòng
                    bridge_rows.append((pid.iloc[i], tp, std, how, raw))
            long_rows.append((pid.iloc[i], tp, o_label, o_status, o_match, a_label, a_status, a_match))
    long_df = pd.DataFrame(long_rows, columns=['patient_hash', 'timepoint', 'oxygen_support', 'oxygen_status', 'oxygen_match',
                                               'antibiotic_regimen', 'antibiotic_status', 'antibiotic_match'])
    bridge = pd.DataFrame(bridge_rows, columns=['patient_hash', 'timepoint', 'drug_name', 'match_method', 'raw_name'])
    assert (long_df.groupby('patient_hash').size() == 3).all(), 'Mỗi bệnh nhân phải có đúng 3 dòng'
    assert len(long_df) == 3 * df[pid_col].nunique()

    drug_names = (bridge.groupby(['drug_name', 'match_method'])
                        .agg(n_rows=('raw_name', 'size'), raw_variants=('raw_name', lambda s: ' | '.join(sorted(set(s)))))
                        .reset_index())

    # Kiểm tra chéo: cột Có/Không "Sử dụng kháng sinh" so với các mốc thực tế
    conflicts = pd.DataFrame(columns=['patient_hash', 'conflict'])
    long_df['flag_conflict'] = ''
    if flag_col:
        said = [('yes' if str(v).strip().lower() in YES_TOKENS else
                 'no' if str(v).strip().lower() in NO_DRUG_TOKENS else 'unknown') for v in df[flag_col]]
        given_any = set(bridge['patient_hash'])
        rows = []
        for p, s in zip(pid, said):
            used = p in given_any
            if s == 'no' and used: rows.append((p, 'flag_no_but_drug_given'))
            if s == 'yes' and not used: rows.append((p, 'flag_yes_but_no_drug'))
        conflicts = pd.DataFrame(rows, columns=['patient_hash', 'conflict'])
        long_df['flag_conflict'] = long_df['patient_hash'].map(conflicts.set_index('patient_hash')['conflict']).fillna('')

    ventilation = None
    if vent_col:
        ventilation = pd.DataFrame({'patient_hash': pid, 'ventilation_hours': pd.to_numeric(df[vent_col], errors='coerce')})
    report = {
        'cột không nhận diện được mốc': oxy_unknown + abx_unknown,
        'tên oxy chưa có trong từ điển': sorted(long_df.loc[long_df.oxygen_match == 'unmatched', 'oxygen_support'].unique()),
        'tên thuốc chưa khớp': sorted(bridge.loc[bridge.match_method == 'unmatched', 'drug_name'].unique()),
        'bí danh thuốc chưa xác nhận': sorted(bridge.loc[bridge.match_method == 'alias_unconfirmed', 'raw_name'].unique()),
        'hàng mâu thuẫn (cột Có/Không so với các mốc)': len(conflicts),
    }
    return long_df, bridge, drug_names, ventilation, conflicts, report

## 2b. Kiểm thử trên dữ liệu giả (chỉ chạy khi chưa có file thật)

Dữ liệu dưới đây tự dựng để kiểm thử logic, không phải dữ liệu của giảng viên, và không được ghi ra đĩa.

In [ ]:
if not CLINICAL_FILE.exists():
    cols = pd.MultiIndex.from_tuples([
        ('Mã bệnh nhân', 'Unnamed: 0_level_1'),
        ('Oxygen support', 'Admission'), ('Oxygen support', '24h'), ('Oxygen support', '48h'),
        ('Sử dụng kháng sinh', 'Unnamed: 4_level_1'),
        ('Antibiotic', 'Admission'), ('Antibiotic', '24h'), ('Antibiotic', '48h'),
        ('Ventilation', 'Hours')])
    demo = pd.DataFrame([
        ['A001', 'Nasal cannula', 'HFNC+ NIV', 'Thở máy', 'Có', 'Etarpenem', 'Etarpenem + Linezolide', 'Linezolide + Metronidazol', 12],
        ['A002', 'Room air', 'Room air', 'Room air', 'Không', 'None', 'None', 'Vancomycin', 0],
        ['A003', 'Mask', np.nan, 'Intubated', 'Có', 'None', 'None', 'None', 96],
        ['A004', 'Oxygen tank xyz', 'RA', 'RA', 'Không', 'Zzzmycin', 'None', 'None', 0]], columns=cols)
    long_df, bridge, drug_names, ventilation, conflicts, report = process(demo)
    print(long_df.to_string()); print(bridge.to_string()); print(drug_names.to_string()); print(report)

    assert len(long_df) == 12 and (long_df.groupby('patient_hash').size() == 3).all()          # đúng 3 dòng mỗi bệnh nhân
    a1 = long_df[(long_df.patient_hash == pseudonymize('A001')) & (long_df.timepoint == '24h')].iloc[0]
    assert a1.antibiotic_regimen == 'Ertapenem + Linezolid' and a1.antibiotic_match == 'alias_unconfirmed'
    assert a1.oxygen_support == 'High-flow nasal cannula + Non-invasive ventilation'           # 'HFNC+ NIV' chuẩn hóa
    b1 = bridge[(bridge.patient_hash == pseudonymize('A001')) & (bridge.timepoint == '24h')]
    assert sorted(b1.drug_name) == ['Ertapenem', 'Linezolid']                                  # bảng cầu: mỗi thuốc một dòng
    assert 'Invasive ventilation' in ' '.join(long_df.oxygen_support.dropna())                 # 'Thở máy' nhận diện được
    assert report['tên thuốc chưa khớp'] == ['Zzzmycin'] and report['tên oxy chưa có trong từ điển'] == ['Oxygen tank xyz']
    assert set(conflicts.conflict) == {'flag_no_but_drug_given', 'flag_yes_but_no_drug'} and len(conflicts) == 3
    assert (long_df.antibiotic_status == 'none').sum() == 7 and (long_df.antibiotic_status == 'missing').sum() == 0
    assert (long_df.oxygen_status == 'missing').sum() == 1                                     # A003 mốc 24h trống
    print('Kiểm thử logic: đạt.')

## 3. Chạy trên file thật

Chỉ chạy khi `CLINICAL_FILE` tồn tại. Không dùng kết quả cho báo cáo nếu báo cáo bên dưới còn mục chưa xử lý.

In [ ]:
if CLINICAL_FILE.exists():
    reader = pd.read_excel if CLINICAL_FILE.suffix.lower() in ('.xlsx', '.xls') else pd.read_csv
    raw_df = reader(CLINICAL_FILE, header=HEADER_ROWS, keep_default_na=False, na_values=[''])
    long_df, bridge, drug_names, ventilation, conflicts, report = process(raw_df)
    print(report)
    ANON.mkdir(parents=True, exist_ok=True)
    long_df.to_csv(ANON / 'clinical_long.csv', index=False)
    bridge.to_csv(ANON / 'clinical_antibiotic_bridge.csv', index=False)
    drug_names.to_csv(ANON / 'clinical_drug_names.csv', index=False)
    conflicts.to_csv(ANON / 'clinical_conflicts.csv', index=False)
    if ventilation is not None:
        ventilation.to_csv(ANON / 'clinical_ventilation.csv', index=False)
    print(long_df.antibiotic_status.value_counts())
else:
    print('Chưa có file lâm sàng thật: bỏ qua bước ghi file.')